# 🧠 Multi-Layer Perceptron (MLP) From Scratch

> Building a neural network from scratch using only NumPy

**Topics covered:**
- Neural network architecture
- Forward propagation
- Backpropagation
- Gradient descent
- Activation functions
- Training on real dataset

## 📚 Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✅ All imports successful!")

## 🔧 Activation Functions

In [ ]:
class ActivationFunctions:
    """
    Common activation functions and their derivatives
    """
    
    @staticmethod
    def sigmoid(x):
        """Sigmoid: σ(x) = 1 / (1 + e^(-x))"""
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    @staticmethod
    def sigmoid_derivative(x):
        """Derivative of sigmoid: σ'(x) = σ(x) * (1 - σ(x))"""
        s = ActivationFunctions.sigmoid(x)
        return s * (1 - s)
    
    @staticmethod
    def relu(x):
        """ReLU: f(x) = max(0, x)"""
        return np.maximum(0, x)
    
    @staticmethod
    def relu_derivative(x):
        """Derivative of ReLU"""
        return (x > 0).astype(float)
    
    @staticmethod
    def tanh(x):
        """Tanh: f(x) = (e^x - e^(-x)) / (e^x + e^(-x))"""
        return np.tanh(x)
    
    @staticmethod
    def tanh_derivative(x):
        """Derivative of tanh"""
        return 1 - np.tanh(x) ** 2
    
    @staticmethod
    def softmax(x):
        """Softmax for output layer"""
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Test activation functions
x = np.array([-2, -1, 0, 1, 2])
print("Input:", x)
print("Sigmoid:", ActivationFunctions.sigmoid(x))
print("ReLU:", ActivationFunctions.relu(x))
print("Tanh:", ActivationFunctions.tanh(x))

## 🏗️ MLP Class - From Scratch

In [ ]:
class MLP:
    """
    Multi-Layer Perceptron implemented from scratch
    
    Architecture:
    Input Layer -> Hidden Layer(s) -> Output Layer
    """
    
    def __init__(self, layer_sizes, activation='relu', learning_rate=0.01):
        """
        Initialize MLP
        
        Args:
            layer_sizes: List of layer dimensions [input, hidden1, hidden2, ..., output]
            activation: 'relu', 'sigmoid', or 'tanh'
            learning_rate: Step size for gradient descent
        """
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.num_layers = len(layer_sizes)
        
        # Select activation function
        if activation == 'relu':
            self.activation = ActivationFunctions.relu
            self.activation_derivative = ActivationFunctions.relu_derivative
        elif activation == 'sigmoid':
            self.activation = ActivationFunctions.sigmoid
            self.activation_derivative = ActivationFunctions.sigmoid_derivative
        elif activation == 'tanh':
            self.activation = ActivationFunctions.tanh
            self.activation_derivative = ActivationFunctions.tanh_derivative
        else:
            raise ValueError(f"Unknown activation: {activation}")
        
        # Initialize weights and biases
        self.weights = []
        self.biases = []
        
        for i in range(self.num_layers - 1):
            # Xavier/Glorot initialization
            limit = np.sqrt(6.0 / (layer_sizes[i] + layer_sizes[i + 1]))
            W = np.random.uniform(-limit, limit, (layer_sizes[i], layer_sizes[i + 1]))
            b = np.zeros((1, layer_sizes[i + 1]))
            
            self.weights.append(W)
            self.biases.append(b)
        
        # Storage for forward pass (needed for backprop)
        self.z_values = []  # Pre-activation values
        self.a_values = []  # Post-activation values
        
        print(f"✅ MLP created: {layer_sizes}")
        print(f"   Total parameters: {sum(w.size + b.size for w, b in zip(self.weights, self.biases))}")
    
    def forward(self, X):
        """
        Forward propagation
        
        Args:
            X: Input data of shape (batch_size, input_features)
        
        Returns:
            Output predictions
        """
        self.z_values = []
        self.a_values = [X]  # a[0] = input
        
        current = X
        
        for i in range(len(self.weights)):
            # Linear transformation: z = a @ W + b
            z = np.dot(current, self.weights[i]) + self.biases[i]
            self.z_values.append(z)
            
            # Apply activation (except last layer)
            if i < len(self.weights) - 1:
                a = self.activation(z)
            else:
                # Output layer: use softmax for classification
                a = ActivationFunctions.softmax(z)
            
            self.a_values.append(a)
            current = a
        
        return current
    
    def backward(self, X, y, output):
        """
        Backpropagation - compute gradients
        
        Args:
            X: Input data
            y: True labels (one-hot encoded)
            output: Predictions from forward pass
        """
        m = X.shape[0]  # Batch size
        
        # Initialize gradients
        weight_gradients = [np.zeros_like(w) for w in self.weights]
        bias_gradients = [np.zeros_like(b) for b in self.biases]
        
        # Output layer error
        delta = output - y  # Cross-entropy derivative
        
        # Backpropagate through layers
        for i in reversed(range(len(self.weights))):
            # Gradients for current layer
            weight_gradients[i] = np.dot(self.a_values[i].T, delta) / m
            bias_gradients[i] = np.sum(delta, axis=0, keepdims=True) / m
            
            # Propagate error to previous layer (if not first layer)
            if i > 0:
                delta = np.dot(delta, self.weights[i].T) * self.activation_derivative(self.z_values[i - 1])
        
        return weight_gradients, bias_gradients
    
    def update_parameters(self, weight_gradients, bias_gradients):
        """Update weights and biases using gradient descent"""
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * weight_gradients[i]
            self.biases[i] -= self.learning_rate * bias_gradients[i]
    
    def compute_loss(self, y_true, y_pred):
        """Cross-entropy loss"""
        m = y_true.shape[0]
        # Add small epsilon to avoid log(0)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        loss = -np.sum(y_true * np.log(y_pred)) / m
        return loss
    
    def compute_accuracy(self, y_true, y_pred):
        """Compute classification accuracy"""
        true_labels = np.argmax(y_true, axis=1)
        pred_labels = np.argmax(y_pred, axis=1)
        return np.mean(true_labels == pred_labels)
    
    def train(self, X, y, epochs=1000, batch_size=32, verbose=100):
        """
        Train the network
        
        Args:
            X: Training data
            y: Training labels (one-hot)
            epochs: Number of training iterations
            batch_size: Mini-batch size
            verbose: Print loss every N epochs
        """
        losses = []
        accuracies = []
        
        for epoch in range(epochs):
            # Mini-batch training
            indices = np.random.permutation(X.shape[0])
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            epoch_loss = 0
            num_batches = 0
            
            for i in range(0, X.shape[0], batch_size):
                X_batch = X_shuffled[i:i + batch_size]
                y_batch = y_shuffled[i:i + batch_size]
                
                # Forward pass
                output = self.forward(X_batch)
                
                # Compute loss
                loss = self.compute_loss(y_batch, output)
                epoch_loss += loss
                num_batches += 1
                
                # Backward pass
                weight_grads, bias_grads = self.backward(X_batch, y_batch, output)
                
                # Update parameters
                self.update_parameters(weight_grads, bias_grads)
            
            # Average loss for epoch
            avg_loss = epoch_loss / num_batches
            losses.append(avg_loss)
            
            # Compute accuracy on full dataset
            predictions = self.forward(X)
            accuracy = self.compute_accuracy(y, predictions)
            accuracies.append(accuracy)
            
            if epoch % verbose == 0:
                print(f"Epoch {epoch:4d} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}")
        
        return losses, accuracies
    
    def predict(self, X):
        """Make predictions"""
        output = self.forward(X)
        return np.argmax(output, axis=1)

# Test the MLP
mlp = MLP(layer_sizes=[2, 10, 5, 2], activation='relu', learning_rate=0.1)
print("\n✅ MLP class created successfully!")

## 📊 Dataset Preparation

In [ ]:
# Create a non-linear classification dataset (moons)
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert labels to one-hot encoding
def one_hot_encode(y, num_classes):
    """Convert integer labels to one-hot vectors"""
    m = len(y)
    one_hot = np.zeros((m, num_classes))
    one_hot[np.arange(m), y] = 1
    return one_hot

y_train_oh = one_hot_encode(y_train, 2)
y_test_oh = one_hot_encode(y_test, 2)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Classes: {np.unique(y)}")

# Visualize the dataset
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='viridis', edgecolors='k', alpha=0.7)
plt.title('Training Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 2, 2)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='viridis', edgecolors='k', alpha=0.7)
plt.title('Test Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.tight_layout()
plt.show()

## 🚀 Training the Network

In [ ]:
# Create and train the MLP
mlp = MLP(
    layer_sizes=[2, 16, 8, 2],  # Input: 2, Hidden: 16, 8, Output: 2
    activation='relu',
    learning_rate=0.5
)

print("\n🚀 Starting training...\n")
losses, accuracies = mlp.train(
    X_train, y_train_oh,
    epochs=500,
    batch_size=32,
    verbose=50
)

print("\n✅ Training complete!")

## 📈 Visualizing Training Progress

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curve
axes[0].plot(losses, color='red', linewidth=2)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(accuracies, color='green', linewidth=2)
axes[1].set_title('Training Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3)

# Decision boundary
def plot_decision_boundary(model, X, y, ax, title):
    """Plot decision boundary"""
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h)
    )
    
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid_points)
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='viridis')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolors='k', alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plot_decision_boundary(mlp, X_train, y_train, axes[2], 'Decision Boundary')

plt.tight_layout()
plt.show()

# Final metrics
train_pred = mlp.predict(X_train)
test_pred = mlp.predict(X_test)

train_acc = np.mean(train_pred == y_train)
test_acc = np.mean(test_pred == y_test)

print(f"\n📊 Final Results:")
print(f"   Training Accuracy: {train_acc:.4f}")
print(f"   Test Accuracy: {test_acc:.4f}")

## 🧪 Comparing Different Activations

In [ ]:
# Compare ReLU vs Sigmoid vs Tanh
activations = ['relu', 'sigmoid', 'tanh']
results = {}

for act in activations:
    print(f"\n🔬 Testing {act.upper()} activation...")
    
    # Create fresh model
    model = MLP(
        layer_sizes=[2, 16, 8, 2],
        activation=act,
        learning_rate=0.5 if act == 'relu' else 0.1
    )
    
    # Train
    losses, accs = model.train(
        X_train, y_train_oh,
        epochs=300,
        batch_size=32,
        verbose=100
    )
    
    # Evaluate
    test_pred = model.predict(X_test)
    test_acc = np.mean(test_pred == y_test)
    
    results[act] = {
        'losses': losses,
        'accuracies': accs,
        'test_acc': test_acc
    }
    
    print(f"   Test Accuracy: {test_acc:.4f}")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for act in activations:
    axes[0].plot(results[act]['losses'], label=act.upper(), linewidth=2)
    axes[1].plot(results[act]['accuracies'], label=act.upper(), linewidth=2)

axes[0].set_title('Loss Comparison', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Comparison Summary:")
for act in activations:
    print(f"   {act.upper():8s}: Test Accuracy = {results[act]['test_acc']:.4f}")

## 🎯 Exercises

1. **Change architecture**: Try different hidden layer sizes `[2, 32, 16, 2]` vs `[2, 8, 4, 2]`
2. **Learning rate**: Test different learning rates (0.01, 0.1, 1.0)
3. **More layers**: Add a 3rd hidden layer `[2, 16, 8, 4, 2]`
4. **Different dataset**: Use `make_classification()` or `make_circles()`
5. **Regularization**: Add L2 regularization to the loss
6. **Momentum**: Implement momentum in gradient descent

## 📚 Key Takeaways

| Concept | What We Learned |
|---------|----------------|
| **Forward Pass** | `z = a @ W + b`, then activation |
| **Backpropagation** | Chain rule to compute gradients |
| **Gradient Descent** | `W = W - lr * dL/dW` |
| **Activation Functions** | ReLU (best), Sigmoid, Tanh |
| **Initialization** | Xavier/Glorot for stable training |
| **Loss Function** | Cross-entropy for classification |

**Next:** Try `02-cnn-basics.ipynb` for Convolutional Neural Networks! 🚀